In [ ]:
import sys; sys.path.append("..")
import pandas as pd
from src.config import CONTAMINANTES, ESTACIONES
from src.limpieza import limpiar_negativos, limpiar_saturados, limpiar_inconsistentes, reindexar_gaps

#### reposicionamiento de estaciones de lotas que pertenecen a enel coronel puede causar confusión

In [ ]:
datos = pd.read_parquet("../data/crudo.parquet")
print(f"Crudo: {len(datos):,} filas")

# La comuna se toma siempre del config (fuente única), no de lo que quedó guardado al descargar
comuna_de = {nombre: comuna for nombre, comuna in ESTACIONES.values()}
datos["comuna"] = datos["estacion"].map(comuna_de)
print(datos.groupby("comuna")["estacion"].unique())

In [ ]:
ancho = datos.pivot_table(
    index=["timestamp", "estacion", "comuna"],
    columns="parametro",
    values="valor",
    aggfunc="mean"
).reset_index()
ancho.columns.name = None
print(f"Ancho: {ancho.shape[0]:,} filas x {ancho.shape[1]} columnas")

In [ ]:
ancho["anio"] = ancho["timestamp"].dt.year
df = ancho[ancho["anio"] >= 2017].copy()
print(f"2017+: {len(df):,} filas")

In [ ]:
contaminantes = [c for c in CONTAMINANTES.keys() if c in df.columns]
df = limpiar_negativos(df, contaminantes)
df = limpiar_saturados(df, contaminantes)
df = limpiar_inconsistentes(df)
df = reindexar_gaps(df)
print(f"\nFinal: {len(df):,} filas")

In [ ]:
df.to_parquet("../data/dataset_limpio.parquet", index=False)
print("dataset_limpio.parquet guardado ✅")